In [1]:

# 2) Install dependencies
!pip -q install -U transformers peft trl accelerate bitsandbytes datasets sentencepiece


In [2]:

# 3) Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:

# 4) Create Week8 paths on Drive
from pathlib import Path

PROJECT_ROOT = Path('/content/drive/MyDrive/Week8')
DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
ADAPTER_DIR = PROJECT_ROOT / 'adapters'
REPORT_DIR = PROJECT_ROOT
LOG_DIR = PROJECT_ROOT / 'logs'

for p in [DATA_DIR, ADAPTER_DIR, LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT =', PROJECT_ROOT)
print('Expecting train file at:', DATA_DIR / 'train.jsonl')
print('Expecting val file at  :', DATA_DIR / 'val.jsonl')


PROJECT_ROOT = /content/drive/MyDrive/Week8
Expecting train file at: /content/drive/MyDrive/Week8/data/processed/train.jsonl
Expecting val file at  : /content/drive/MyDrive/Week8/data/processed/val.jsonl


In [3]:

# 5) Basic checks
from pathlib import Path

train_path = DATA_DIR / 'train.jsonl'
val_path = DATA_DIR / 'val.jsonl'

assert train_path.exists(), f'Missing: {train_path}'
assert val_path.exists(), f'Missing: {val_path}'

print('Found train:', train_path)
print('Found val  :', val_path)


Found train: /content/drive/MyDrive/Week8/data/processed/train.jsonl
Found val  : /content/drive/MyDrive/Week8/data/processed/val.jsonl


In [12]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
MAX_SEQ_LENGTH = 512

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

model.config.use_cache = False

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [13]:
# 7) Imports and constants
import json
import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig
from transformers import set_seed

SEED = 42
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LEARNING_RATE = 2e-4
PER_DEVICE_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
NUM_EPOCHS = 3
USE_4BIT = True

set_seed(SEED)

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Torch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


In [14]:

# 8) Load dataset
raw_datasets = load_dataset(
    'json',
    data_files={
        'train': str(train_path),
        'validation': str(val_path),
    }
)
raw_datasets


DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 1500
    })
    validation: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 600
    })
})

In [15]:
# 9) Convert instruction dataset -> single training text
# Expected format from Day 1:
# {"instruction":"...", "input":"...", "output":"..."}

def format_example(example):
    instruction = (example.get("instruction") or "").strip()
    inp = (example.get("input") or "").strip()
    out = (example.get("output") or "").strip()

    if inp:
        text = (
            f"### Instruction:\n{instruction}\n\n"
            f"### Input:\n{inp}\n\n"
            f"### Response:\n{out}"
        )
    else:
        text = (
            f"### Instruction:\n{instruction}\n\n"
            f"### Response:\n{out}"
        )

    return {"text": text}

train_dataset = raw_datasets["train"].map(format_example)
val_dataset = raw_datasets["validation"].map(format_example)

print(train_dataset[0]["text"][:800])

### Instruction:
Extract the required medical entities from the text. Return only valid JSON.

### Input:
Olivia Clark,
oliviaclark@example.com
12121 Oak Lane, Seaside, FL 32459,
850-555-1212, United States

Relationship to XYZ Pharma Inc.: Patient
Reason for contacting: Adverse Event

Message: Hi, I've been on Complera for a while and have started noticing symptoms of a new infection, like fever and weight loss. Could this be related to the medication? Best, Olivia Clark

### Response:
{"drug_name": "Complera", "adverse_events": ["symptoms of a new infection", "fever", "weight loss"]}


In [16]:

# 10) Tokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = 'right'
print('pad_token:', tokenizer.pad_token)
print('eos_token:', tokenizer.eos_token)


pad_token: </s>
eos_token: </s>


In [17]:
# 12) LoRA config
peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

trainable_params = 0
total_params = 0
for _, param in model.named_parameters():
    total_params += param.numel()
    if param.requires_grad:
        trainable_params += param.numel()

trainable_pct = 100 * trainable_params / total_params
print(f"Trainable params: {trainable_params:,}")
print(f"Total params    : {total_params:,}")
print(f"Trainable %     : {trainable_pct:.4f}%")

trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338
Trainable params: 12,615,680
Total params    : 628,221,952
Trainable %     : 2.0082%


In [18]:
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = str(ADAPTER_DIR)

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    logging_steps=10,
    save_steps=100,
    eval_steps=100,
    eval_strategy="steps",
    save_strategy="steps",
    report_to="none",
    fp16=False,
    bf16=False,
    gradient_checkpointing=True,
    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

In [19]:

# 15) Start training
train_result = trainer.train()
train_result


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss,Validation Loss
100,1.254484,1.179805
200,1.076577,1.174332


TrainOutput(global_step=282, training_loss=1.1910555802338512, metrics={'train_runtime': 1750.071, 'train_samples_per_second': 2.571, 'train_steps_per_second': 0.161, 'total_flos': 1.2731753688367104e+16, 'train_loss': 1.1910555802338512})

In [21]:
# 16) Collect evaluation info from training logs
eval_rows = [x for x in trainer.state.log_history if "eval_loss" in x]
eval_rows

[{'eval_loss': 1.1798053979873657,
  'eval_runtime': 65.6984,
  'eval_samples_per_second': 9.133,
  'eval_steps_per_second': 2.283,
  'eval_entropy': 1.1883414713541667,
  'eval_num_tokens': 438983.0,
  'eval_mean_token_accuracy': 0.717604192495346,
  'epoch': 1.064,
  'step': 100},
 {'eval_loss': 1.1743316650390625,
  'eval_runtime': 65.545,
  'eval_samples_per_second': 9.154,
  'eval_steps_per_second': 2.289,
  'eval_entropy': 1.1562923177083333,
  'eval_num_tokens': 877184.0,
  'eval_mean_token_accuracy': 0.7192771629492442,
  'epoch': 2.128,
  'step': 200},
 {'eval_loss': 1.1695263385772705,
  'eval_runtime': 66.2218,
  'eval_samples_per_second': 9.06,
  'eval_steps_per_second': 2.265,
  'eval_entropy': 1.1392643229166666,
  'eval_num_tokens': 1238037.0,
  'eval_mean_token_accuracy': 0.719408635298411,
  'epoch': 3.0,
  'step': 282}]

In [22]:

# 17) Save adapter in standard PEFT format
trainer.model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))

print('Saved PEFT adapter files to:', ADAPTER_DIR)
print('Files:', [p.name for p in ADAPTER_DIR.iterdir()])


Saved PEFT adapter files to: /content/drive/MyDrive/Week8/adapters
Files: ['README.md', 'checkpoint-100', 'checkpoint-200', 'checkpoint-282', 'adapter_model.safetensors', 'adapter_config.json', 'chat_template.jinja', 'tokenizer_config.json', 'tokenizer.json']


In [23]:

# 18) Also save adapter weights as adapter_model.bin
from peft import get_peft_model_state_dict

adapter_state = get_peft_model_state_dict(trainer.model)
bin_path = ADAPTER_DIR / 'adapter_model.bin'
torch.save(adapter_state, bin_path)
print('Saved:', bin_path)
print('Size (MB):', round(bin_path.stat().st_size / (1024 * 1024), 2))


Saved: /content/drive/MyDrive/Week8/adapters/adapter_model.bin
Size (MB): 24.17


In [25]:
# 19) Save training summary JSON for report use
import json

eval_rows = [x for x in trainer.state.log_history if "eval_loss" in x]
last_eval = eval_rows[-1] if eval_rows else {}

summary = {
    "model_name": MODEL_NAME,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "learning_rate": LEARNING_RATE,
    "per_device_batch_size": PER_DEVICE_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "epochs": NUM_EPOCHS,
    "load_in_4bit": USE_4BIT,
    "max_seq_length": MAX_SEQ_LENGTH,
    "trainable_params": int(trainable_params),
    "total_params": int(total_params),
    "trainable_pct": float(trainable_pct),
    "train_loss": float(train_result.metrics.get("train_loss", 0.0)),
    "train_runtime": float(train_result.metrics.get("train_runtime", 0.0)),
    "train_steps_per_second": float(train_result.metrics.get("train_steps_per_second", 0.0)),
    "train_samples_per_second": float(train_result.metrics.get("train_samples_per_second", 0.0)),
    "eval_loss": float(last_eval["eval_loss"]) if "eval_loss" in last_eval and last_eval["eval_loss"] is not None else None,
    "eval_step": int(last_eval["step"]) if "step" in last_eval else None,
}

summary_path = PROJECT_ROOT / "training_summary_day2.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Saved summary to:", summary_path)
summary

Saved summary to: /content/drive/MyDrive/Week8/training_summary_day2.json


{'model_name': 'TinyLlama/TinyLlama-1.1B-Chat-v1.0',
 'lora_r': 16,
 'lora_alpha': 32,
 'lora_dropout': 0.05,
 'learning_rate': 0.0002,
 'per_device_batch_size': 4,
 'gradient_accumulation_steps': 4,
 'epochs': 3,
 'load_in_4bit': True,
 'max_seq_length': 512,
 'trainable_params': 12615680,
 'total_params': 628221952,
 'trainable_pct': 2.00815650580768,
 'train_loss': 1.1910555802338512,
 'train_runtime': 1750.071,
 'train_steps_per_second': 0.161,
 'train_samples_per_second': 2.571,
 'eval_loss': 1.1695263385772705,
 'eval_step': 282}

In [26]:

# 20) Quick inference sanity check from adapter
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
base_model.config.use_cache = True

ft_model = PeftModel.from_pretrained(base_model, str(ADAPTER_DIR))
ft_model.eval()

prompt = """### Instruction:
Answer this medical question truthfully.

### Input:
What is the likely diagnosis in a patient with fever, productive cough, and lobar consolidation on chest X-ray?

### Response:
"""

inputs = tokenizer(prompt, return_tensors='pt').to(ft_model.device)
with torch.no_grad():
    outputs = ft_model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
Answer this medical question truthfully.

### Input:
What is the likely diagnosis in a patient with fever, productive cough, and lobar consolidation on chest X-ray?

### Response:
The likely diagnosis in a patient with fever, productive cough, and lobar consolidation on chest X-ray is tuberculosis.
